# spark.read — read a CSV (try it live)

Runnable companion to the note: **[DataFrameReader](https://ravi-writes.pages.dev/notes/abinitio-to-pyspark/core-operations/dataframereader)**.

`spark.read` is Ab Initio's **Input File / Input Table**. This example needs a real file, so first we write one with **plain Python + Faker** (standing in for a file from some upstream system), then the PySpark program reads it. Run the cells top to bottom.

## The Ab Initio equivalent

The CSV is created *outside* Spark, like a file that lands from an upstream feed. `spark.read` is the **Input File** that pulls it into a DataFrame.

> **Why plain Python?** Using Spark to make the file we then read with Spark would be circular. Plain Python gives a true external input — one normal `customers.csv` under `/content/ravi-writes/data/input/` (Spark outputs go under `/content/ravi-writes/data/output/`). (When *Spark itself* writes, it makes a **directory of `part-*` files** — that's the `df.write` / Output File story.) `/content` is Colab's **ephemeral** scratch disk.

```text
  Python + Faker ──(writes)──►  …/data/input/customers.csv   (external file, not Spark)
                                        │
                                        ▼
                        Input File ──(spark.read)──►  DataFrame
```

## 0. Install PySpark & Faker

Colab ships with neither — one `pip install` sets up both in the runtime (a few seconds).

In [ ]:
!pip install -q pyspark faker

## 1. Create the input file (plain Python + Faker)

No Spark here — just ordinary Python writing a normal CSV. This stands in for a file you received from elsewhere. The seed makes it reproducible.

In [ ]:
import csv, os, random
from faker import Faker

# workspace layout: inputs under data/input, Spark outputs under data/output
INPUT_PATH = "/content/ravi-writes/data/input/customers.csv"
os.makedirs(os.path.dirname(INPUT_PATH), exist_ok=True)

fake = Faker()
Faker.seed(42)
random.seed(42)          # reproducible — same rows every run

with open(INPUT_PATH, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "name", "signup_date", "balance"])
    for i in range(1, 9):        # 8 customers
        writer.writerow([
            i,
            fake.name(),
            fake.date_between(start_date="-2y", end_date="today").isoformat(),
            round(random.uniform(0, 500), 2),
        ])

# it's just plain text on disk — peek at it
print(open(INPUT_PATH).read())

## 2. Imports & SparkSession

`SparkSession.builder...getOrCreate()` is the builder object — see [The Builder Object](https://ravi-writes.pages.dev/notes/python-for-spark/).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType,
)

spark = SparkSession.builder.appName("read-demo").getOrCreate()

## 3. Read it with spark.read

With no schema, every CSV column comes in as a **string**. Pin the types with an explicit schema — the Ab Initio DML parallel.

In [ ]:
# no schema -> every column is a string
raw = spark.read.option("header", True).csv(INPUT_PATH)
raw.printSchema()

# explicit schema -> exact types (the Ab Initio DML)
schema = StructType([
    StructField("id",          IntegerType(), nullable=True),
    StructField("name",        StringType(),  nullable=True),
    StructField("signup_date", StringType(),  nullable=True),
    StructField("balance",     DoubleType(),  nullable=True),
])

df = spark.read.schema(schema).option("header", True).csv(INPUT_PATH)
df.show()
df.printSchema()

## Your turn

1. **Let Spark guess** — read again with `.option("inferSchema", True)` (no `.schema(...)`), then `printSchema()`. What did it get right? (`inferSchema` is Spark's convenience; Ab Initio always needs its DML up front.)
2. **The generic form** — rewrite the typed read as `spark.read.format("csv").schema(schema).option("header", True).load(INPUT_PATH)` and confirm it matches.
3. **Scale up the input** — bump the loop to `range(1, 1001)` for 1,000 customers, re-run, and try `df.count()`. Same program, bigger file.

Try them yourself first, then reveal the solutions below.

In [ ]:
# 1. Let Spark infer the types
inferred = spark.read.option("header", True).option("inferSchema", True).csv(INPUT_PATH)
inferred.printSchema()

# 2. Generic form — format + load — same result as the .csv(...) shortcut
df2 = spark.read.format("csv").schema(schema).option("header", True).load(INPUT_PATH)
df2.show()

# 3. Same program, bigger file
print("rows:", df.count())